In [ ]:
#parse out frames from videos
#ChatGPT__________ONLYUSEONCE
import cv2
import os
import glob

def extract_frames(video_path, output_dir, every_n_frames=1):
    """
    Extracts frames from a video and saves them as JPGs.
    """
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    print(f"🎥 Extracting from: {video_path}")
    print(f"Total frames: {frame_count}, FPS: {fps}")

    i = 0
    saved = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if i % every_n_frames == 0:
            frame_name = f"frame_{i:06d}.jpg"
            frame_path = os.path.join(output_dir, frame_name)
            cv2.imwrite(frame_path, frame)
            saved += 1

        i += 1

    cap.release()
    print(f"✅ Saved {saved} frames to {output_dir}")

# Example usage:
#extract_frames("C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\YOLO Format\IMG_7918\IMG_7918_dusty.mov", "video1/images", every_n_frames=1)

video_dir = r"C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Raw Videos"  # <-- change this
video_extensions = ("*.mp4", "*.mov", "*.avi", "*.mkv")

for ext in video_extensions:
    for video_path in glob.glob(os.path.join(video_dir, ext)):
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        output_dir = os.path.join(video_dir, video_name, "images")

        extract_frames(video_path, output_dir, every_n_frames=1)


🎥 Extracting from: C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Raw Videos\IMG_7919_dusty.mov
Total frames: 50, FPS: 30.0
✅ Saved 50 frames to C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Raw Videos\IMG_7919_dusty\images
🎥 Extracting from: C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Raw Videos\IMG_7942_dusty.mov
Total frames: 57, FPS: 30.0
✅ Saved 57 frames to C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Raw Videos\IMG_7942_dusty\images
🎥 Extracting from: C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Raw Videos\IMG_7943_khem.mov
Total frames: 51, FPS: 30.0
✅ Saved 51 frames to C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Raw Videos\IMG_7943_khem\images
🎥 Extracting from: C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Raw Videos\IMG_7944_khem.mov
Total frames: 54, FPS: 30.0
✅ Saved 54 frames to C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Raw Videos\IMG_7944_khem\images
🎥 Extracting fro

In [4]:
import os
import shutil
import glob

# Paths
frames_dir = r"C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Frames"
annotations_dir = r"C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\CVAT Annotations"
project_dir = r"C:\\Users\jared\OneDrive\Grad year Two\Forecasting\Project"

# Make sure target directory exists
os.makedirs(project_dir, exist_ok=True)

# Loop over each annotation file
for ann_path in glob.glob(os.path.join(annotations_dir, "*.xml")):
    base_name = os.path.splitext(os.path.basename(ann_path))[0]
    print(f"🧩 Processing: {base_name}")

    # Make video subfolder inside project/
    video_dir = os.path.join(project_dir, base_name)
    os.makedirs(video_dir, exist_ok=True)

    # Copy annotation file
    dest_ann = os.path.join(video_dir, "annotations.xml")
    shutil.copy(ann_path, dest_ann)

    # Find matching frames folder
    src_frames = os.path.join(frames_dir, base_name, "images")
    dest_frames = os.path.join(video_dir, "images")

    if os.path.exists(src_frames):
        shutil.copytree(src_frames, dest_frames, dirs_exist_ok=True)
        print(f"✅ Copied frames from {src_frames}")
    else:
        print(f"⚠️ No frame folder found for {base_name}")


🧩 Processing: IMG_7943
✅ Copied frames from C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Frames\IMG_7943\images
🧩 Processing: IMG_7944
✅ Copied frames from C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Frames\IMG_7944\images
🧩 Processing: IMG_7997
✅ Copied frames from C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Frames\IMG_7997\images
🧩 Processing: IMG_7998
✅ Copied frames from C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Frames\IMG_7998\images
🧩 Processing: IMG_7999
✅ Copied frames from C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Frames\IMG_7999\images
🧩 Processing: IMG_8027
✅ Copied frames from C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Frames\IMG_8027\images
🧩 Processing: IMG_8029
✅ Copied frames from C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Frames\IMG_8029\images
🧩 Processing: IMG_8030
✅ Copied frames from C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\Frames\IMG_8030

In [2]:
import os
import xml.etree.ElementTree as ET
import cv2  # ✅ for reading image dimensions

def convert_cvat_video_to_yolo(xml_path, images_dir, labels_dir, class_map):
    """
    Converts a CVAT video annotation XML (with <track> and <box>) to YOLO labels.
    Auto-detects image size, normalizes label capitalization, and handles 'moving' attributes.
    """
    os.makedirs(labels_dir, exist_ok=True)

    tree = ET.parse(xml_path)
    root = tree.getroot()

    for track in root.findall("track"):
        # Normalize the label name (e.g., Baseball → baseball)
        base_label = track.get("label").strip().lower()

        # Loop over all <box> entries in this track
        for box in track.findall("box"):
            frame_num = int(box.get("frame"))
            image_name = f"frame_{frame_num:06d}.jpg"
            image_path = os.path.join(images_dir, image_name)
            if not os.path.exists(image_path):
                continue  # Skip if image not found

            # Detect image dimensions dynamically
            img = cv2.imread(image_path)
            if img is None:
                continue  # skip if unreadable
            img_height, img_width = img.shape[:2]

            # Check for 'moving' attribute
            moving_attr = None
            for attr in box.findall("attribute"):
                attr_name = attr.get("name").strip().lower()
                attr_value = attr.text.strip().lower()
                if "moving" in attr_name and attr_value == "true":
                    moving_attr = True
                    break

            # Choose label name
            label_name = base_label
            if moving_attr:
                label_name = f"moving_{base_label}"

            label_name = label_name.lower()
            if label_name not in class_map:
                continue
            class_id = class_map[label_name]

            # Bounding box
            xtl = float(box.get("xtl"))
            ytl = float(box.get("ytl"))
            xbr = float(box.get("xbr"))
            ybr = float(box.get("ybr"))
            width = xbr - xtl
            height = ybr - ytl
            x_center = xtl + width / 2
            y_center = ytl + height / 2

            # Normalize coordinates
            x_center /= img_width
            y_center /= img_height
            width /= img_width
            height /= img_height

            # Write YOLO label file (append in case multiple boxes per frame)
            label_path = os.path.join(labels_dir, os.path.splitext(image_name)[0] + ".txt")
            with open(label_path, "a") as f:
                f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

    print(f"✅ Converted {os.path.basename(xml_path)} → YOLO format.")


# === Batch conversion for all videos ===

project_dir = r"C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project"

# Define unified class mapping — all lowercase
class_map = {
    "baseball": 0,
    "moving_baseball": 1
}

for video_folder in os.listdir(project_dir):
    video_path = os.path.join(project_dir, video_folder)
    xml_path = os.path.join(video_path, "annotations.xml")
    images_dir = os.path.join(video_path, "images")
    labels_dir = os.path.join(video_path, "labels")

    if os.path.exists(xml_path) and os.path.exists(images_dir):
        print(f"\n🎬 Processing {video_folder}")
        convert_cvat_video_to_yolo(xml_path, images_dir, labels_dir, class_map)
    else:
        print(f"⚠️ Skipping {video_folder}: missing XML or images folder")



🎬 Processing IMG_7943
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_7944
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_7997
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_7998
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_7999
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_8027
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_8029
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_8030
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_8060
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_8061
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_8062
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_8063
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_8121
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_8122
✅ Converted annotations.xml → YOLO format.

🎬 Processing IMG_8123
✅ Converted annotations.xml → YOLO format.

🎬 Process

In [9]:
import os
import shutil
from sklearn.model_selection import train_test_split

project_dir = r"C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project"
output_dir = os.path.join(project_dir, "yolo_data")

# Create YOLO folders
for sub in ["images/train", "images/val", "labels/train", "labels/val"]:
    os.makedirs(os.path.join(output_dir, sub), exist_ok=True)

# Collect all (image, label, video_name)
image_label_pairs = []

for video_folder in os.listdir(project_dir):
    video_path = os.path.join(project_dir, video_folder)
    if not os.path.isdir(video_path):
        continue
    img_dir = os.path.join(video_path, "images")
    lbl_dir = os.path.join(video_path, "labels")
    if not (os.path.exists(img_dir) and os.path.exists(lbl_dir)):
        continue
    
    for file in os.listdir(img_dir):
        if file.endswith(".jpg"):
            img_path = os.path.join(img_dir, file)
            lbl_path = os.path.join(lbl_dir, file.replace(".jpg", ".txt"))
            if os.path.exists(lbl_path):
                image_label_pairs.append((img_path, lbl_path, video_folder))

print(f"Found {len(image_label_pairs)} image-label pairs total.")

# Split into train/val
train_pairs, val_pairs = train_test_split(image_label_pairs, test_size=0.2, random_state=42)

def copy_pairs(pairs, split):
    for img_path, lbl_path, vid_name in pairs:
        base_name = f"{vid_name}_{os.path.basename(img_path)}"
        new_img_path = os.path.join(output_dir, "images", split, base_name)
        new_lbl_path = os.path.join(output_dir, "labels", split, base_name.replace(".jpg", ".txt"))
        
        shutil.copy(img_path, new_img_path)
        shutil.copy(lbl_path, new_lbl_path)

copy_pairs(train_pairs, "train")
copy_pairs(val_pairs, "val")

print(f"✅ Copied {len(train_pairs)} train and {len(val_pairs)} val pairs.")


Found 1178 image-label pairs total.
✅ Copied 942 train and 236 val pairs.


In [10]:
import yaml
import os

output_dir = os.path.join(r"C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project", "yolo_data")

data = {
    "train": os.path.join(output_dir, "images/train").replace("\\", "/"),
    "val": os.path.join(output_dir, "images/val").replace("\\", "/"),
    "nc": 2,  # number of classes
    "names": ["baseball", "moving_baseball"]  # adjust if you have more classes
}

yaml_path = os.path.join(output_dir, "data.yaml")

with open(yaml_path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print(f"✅ data.yaml created at: {yaml_path}")


✅ data.yaml created at: C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\yolo_data\data.yaml


In [12]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(data="C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\yolo_data\data.yaml",
            epochs = 30,
            imgsz = 640,
            batch = 8)

<>:5: SyntaxWarning: invalid escape sequence '\j'
<>:5: SyntaxWarning: invalid escape sequence '\j'
C:\Users\jared\AppData\Local\Temp\ipykernel_36116\157762141.py:5: SyntaxWarning: invalid escape sequence '\j'
  model.train(data="C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\yolo_data\data.yaml",


100%|██████████| 6.25M/6.25M [00:00<00:00, 22.4MB/s]


New https://pypi.org/project/ultralytics/8.3.222 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.159  Python-3.12.7 torch-2.7.1+cpu CPU (Intel Core(TM) Ultra 5 125U)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\yolo_data\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=tra

100%|██████████| 755k/755k [00:00<00:00, 8.80MB/s]

Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             


...

                   all        107        726      0.447        0.3      0.262     0.0981



30 epochs completed in 4.596 hours.
Optimizer stripped from runs\detect\train\weights\last.pt, 6.2MB
Optimizer stripped from runs\detect\train\weights\best.pt, 6.2MB

Validating runs\detect\train\weights\best.pt...
Ultralytics 8.3.159  Python-3.12.7 torch-2.7.1+cpu CPU (Intel Core(TM) Ultra 5 125U)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 7/7 [00:37<00:00,  5.39s/it]


                   all        107        726      0.447        0.3      0.261     0.0981
              baseball        100        513      0.568      0.304      0.331      0.133
       moving_baseball         39        213      0.326      0.296      0.192     0.0636
Speed: 3.6ms preprocess, 134.7ms inference, 0.0ms loss, 2.3ms postprocess per image
Results saved to runs\detect\train


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000020D2B4859D0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0

In [2]:
import shutil
import os

source = r"runs/detect/train/weights/best.pt"  # adjust if train1/, train2/, etc.
destination = r"C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project"

os.makedirs(os.path.dirname(destination), exist_ok=True)
shutil.copy(source, destination)

print(f"✅ Model saved at: {destination}")


✅ Model saved at: C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project


In [3]:
from ultralytics import YOLO

model = YOLO(r"C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\best.pt")

# Test on a new video or image folder
results = model.predict(source=r"C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project_Extra\Raw Videos\IMG_7919_dusty.mov", save=True)



WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/50) C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project_Extra\Raw Videos\IMG_7919_dusty.mov: 384x640 (no detections), 115.3ms
video 1/1 (frame 2/50) C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project_Extra\Raw Videos\IMG_7919_dusty.mov: 384x640 (no detections), 97.9ms
video 1/1 (frame 3/50) C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project_Extra\Raw Videos\IMG_7919_dusty.mov: 384x640 (no detections), 91.2ms
video 1/

In [2]:
#stuff for assignment 3:
#zach's code----------------
#doesn't work bc of mislabeled directory
import os
import xml.etree.ElementTree as ET
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms

class CustomBaseball(Dataset):
    def __init__(self, frames_root, xml_folder, transform=None):
        self.frames_root = frames_root
        self.annotations = self.parse_cvat_folder(xml_folder)
        self.transform = transform or transforms.Compose([
            transforms.Resize((128,128)),
            transforms.ToTensor()
        ])

        # Filter out missing frames
        valid_annotations = []
        for ann in self.annotations:
            video_folder = os.path.join(self.frames_root, ann['video_file'])
            if not os.path.exists(video_folder):
                # Try to match folder ignoring case
                folders = os.listdir(self.frames_root)
                match = [f for f in folders if ann['video_file'].lower() in f.lower()]
                if match:
                    video_folder = os.path.join(self.frames_root, match[0])
                    ann['video_file'] = match[0]
                else:
                    continue

            frame_path = os.path.join(video_folder, f"frame_{ann['frame']:04d}.jpg")
            if os.path.exists(frame_path):
                valid_annotations.append(ann)

        self.annotations = valid_annotations
        print(f"Total valid annotations: {len(self.annotations)}")

    def parse_cvat_folder(self, xml_folder):
        all_annotations = []
        for file in os.listdir(xml_folder):
            if file.endswith(".xml"):
                xml_path = os.path.join(xml_folder, file)
                tree = ET.parse(xml_path)
                root = tree.getroot()
                for track in root.findall('track'):
                    label = track.attrib['label']
                    track_id = int(track.attrib['id'])
                    for box in track.findall('box'):
                        frame = int(box.attrib['frame'])
                        outside = int(box.attrib['outside'])
                        if outside == 1:
                            continue
                        xtl = float(box.attrib['xtl'])
                        ytl = float(box.attrib['ytl'])
                        xbr = float(box.attrib['xbr'])
                        ybr = float(box.attrib['ybr'])
                        all_annotations.append({
                            'track_id': track_id,
                            'label': label,
                            'frame': frame,
                            'bbox': [xtl, ytl, xbr, ybr],
                            'video_file': file.replace('.xml','')
                        })
        return all_annotations

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        ann = self.annotations[idx]
        video_folder = os.path.join(self.frames_root, ann['video_file'])
        frame_filename = f"frame_{ann['frame']:04d}.jpg"
        img_path = os.path.join(video_folder, frame_filename)

        image = Image.open(img_path).convert("RGB")
        xtl, ytl, xbr, ybr = ann['bbox']
        cropped = image.crop((xtl, ytl, xbr, ybr))
        image_tensor = self.transform(cropped)
        label = 1  # all baseballs
        return image_tensor, label
    


In [ ]:
import os
import xml.etree.ElementTree as ET
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms

class CustomBaseballDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform

        # Collect all (image_path, boxes, labels) tuples
        self.samples, self.label_map = self._parse_all_annotations()

        print(f"Loaded {len(self.samples)} samples with labels: {self.label_map}")

    def _parse_all_annotations(self):
        all_samples = []
        label_map = {} 
        next_label_id = 0

        for video_folder in os.listdir(self.root_dir):
            video_path = os.path.join(self.root_dir, video_folder)
            if not os.path.isdir(video_path):
                continue

            xml_path = os.path.join(video_path, "annotations.xml")
            images_dir = os.path.join(video_path, "images")

            if not os.path.exists(xml_path):
                continue

            tree = ET.parse(xml_path) #ET is very very useful for xml files
            root = tree.getroot()

            for track in root.findall("track"):
                label = track.attrib["label"]

                if label not in label_map:
                    label_map[label] = next_label_id
                    next_label_id += 1

                for box in track.findall("box"): #used AI for this part
                    frame_id = int(box.attrib["frame"])
                    xtl = float(box.attrib["xtl"])
                    ytl = float(box.attrib["ytl"])
                    xbr = float(box.attrib["xbr"])
                    ybr = float(box.attrib["ybr"])

                    frame_name = f"frame_{frame_id:06d}.jpg"
                    frame_path = os.path.join(images_dir, frame_name)

                    if os.path.exists(frame_path):
                        all_samples.append({
                            "image_path": frame_path,
                            "boxes": torch.tensor([[xtl, ytl, xbr, ybr]], dtype=torch.float32),
                            "labels": torch.tensor([label_map[label]], dtype=torch.int64),
                        })


        return all_samples, label_map

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        image_path = sample["image_path"]
        labels = sample["labels"] 

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)

        return image, labels.squeeze(0)  #google told me to do this


In [ ]:
dataset = CustomBaseballDataset(root_dir = "C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project")
print("Number of samples found:", len(dataset))



<>:1: SyntaxWarning: invalid escape sequence '\j'
<>:1: SyntaxWarning: invalid escape sequence '\j'
C:\Users\jared\AppData\Local\Temp\ipykernel_11560\1748106906.py:1: SyntaxWarning: invalid escape sequence '\j'
  dataset = CustomBaseballDataset(root_dir = "C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project")


⚠️ Warning: No XML found in C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\.git
⚠️ Warning: No XML found in C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\yolo_data
Loaded 19340 samples with labels: {'baseball': 0, 'Baseball': 1}
Number of samples found: 19340


C:\Users\jared\AppData\Local\Temp\ipykernel_11560\1748106906.py:1: SyntaxWarning: invalid escape sequence '\j'
  dataset = CustomBaseballDataset(root_dir = "C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project")


NameError: name 'DataLoader' is not defined

In [ ]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


transform = transforms.Compose([
    transforms.Resize((224, 224)), #dampens resolution. 
    transforms.ToTensor()           
])
dataset = CustomBaseballDataset(root_dir = "C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project",
                                transform=transform)

dataset_size = len(dataset)
train_size = int(dataset_size * 0.8)
remaining_size = dataset_size - train_size

lengths = [train_size, remaining_size]


torch.manual_seed(67)
train_dataset, val_dataset = random_split(dataset, lengths)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


<>:10: SyntaxWarning: invalid escape sequence '\j'
<>:10: SyntaxWarning: invalid escape sequence '\j'
C:\Users\jared\AppData\Local\Temp\ipykernel_11560\1268025383.py:10: SyntaxWarning: invalid escape sequence '\j'
  dataset = CustomBaseballDataset(root_dir = "C:\\Users\jared\OneDrive\Grad Year Two\Forecasting\Project",


⚠️ Warning: No XML found in C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\.git
⚠️ Warning: No XML found in C:\Users\jared\OneDrive\Grad Year Two\Forecasting\Project\yolo_data
Loaded 19340 samples with labels: {'baseball': 0, 'Baseball': 1}


In [4]:
# For reading data
import pandas as pd
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# For visualizing
import plotly.express as px

# For model building
import torch
import torch.nn as nn
import torch.nn.functional as F
class FirstNet(nn.Module):
    def __init__(self):
      # We define the components of our model here
      super(FirstNet, self).__init__()
      # Function to flatten our image
      self.flatten = nn.Flatten()
      # Create the sequence of our network
      self.linear_relu_model = nn.Sequential(
            # Add a linear output layer w/ 10 perceptrons
            nn.LazyLinear(10),
        )

    def forward(self, x):
      # We construct the sequencing of our model here
      x = self.flatten(x)
      # Pass flattened images through our sequence
      output = self.linear_relu_model(x)

      # Return the evaluations of our ten 
      #   classes as a 10-dimensional vector
      return output

# Create an instance of our model
model = FirstNet()

In [9]:
# Define some training parameters
learning_rate = 1e-2
batch_size = 16
epochs = 1

# Define our loss function
#   This one works for multiclass problems
loss_fn = nn.CrossEntropyLoss()
# Build our optimizer with the parameters from
#   the model we defined, and the learning rate
#   that we picked
optimizer = torch.optim.SGD(model.parameters(),
     lr=learning_rate)

In [14]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode
    # important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    # Loop over batches via the dataloader
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation and looking for improved gradients
        loss.backward()
        optimizer.step()
        # Zeroing out the gradient (otherwise they are summed)
        #   in preparation for next round
        optimizer.zero_grad()

        # Print progress update every few loops
        if batch % 10 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode
    # important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    print("Running test loop...")  # indicator that testing has started

    # Evaluating the model with torch.no_grad() ensures
    # that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations
    # and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for batch, (X, y) in enumerate(dataloader):
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            if batch % 10 == 0:
                print(f"  [Batch {batch+1}/{num_batches}] running...")

    # Printing some output after a testing round
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {
        (100*correct):>0.1f}%, Avg loss: {
            test_loss:>8f} \n")
    
# Need to repeat the training process for each epoch.
#   In each epoch, the model will eventually see EVERY
#   observations in the data
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_loader, model, loss_fn, optimizer)
    test_loop(val_loader, model, loss_fn=loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.241719  [   32/15472]
loss: 7.871941  [  352/15472]
loss: 2.126531  [  672/15472]
loss: 9.665871  [  992/15472]
loss: 0.284327  [ 1312/15472]
loss: 1.080111  [ 1632/15472]
loss: 1.065437  [ 1952/15472]
loss: 7.775752  [ 2272/15472]
loss: 0.061055  [ 2592/15472]
loss: 12.777290  [ 2912/15472]
loss: 2.871413  [ 3232/15472]
loss: 0.064837  [ 3552/15472]
loss: 0.401102  [ 3872/15472]
loss: 0.308315  [ 4192/15472]
loss: 2.338362  [ 4512/15472]
loss: 0.439215  [ 4832/15472]
loss: 3.041143  [ 5152/15472]
loss: 1.000544  [ 5472/15472]
loss: 1.646699  [ 5792/15472]
loss: 0.000008  [ 6112/15472]
loss: 1.819247  [ 6432/15472]
loss: 2.619186  [ 6752/15472]
loss: 9.627098  [ 7072/15472]
loss: 0.031036  [ 7392/15472]
loss: 1.180494  [ 7712/15472]
loss: 0.623313  [ 8032/15472]
loss: 4.121930  [ 8352/15472]
loss: 5.157021  [ 8672/15472]
loss: 1.238694  [ 8992/15472]
loss: 0.570535  [ 9312/15472]
loss: 6.794026  [ 9632/15472]
loss: 0.736213  [ 9952/15472]

In [7]:
#testing
for i in range(3):
    print(dataset[i])

(tensor([[[0.2392, 0.2275, 0.2118,  ..., 0.2314, 0.2863, 0.3137],
         [0.2353, 0.2275, 0.2157,  ..., 0.2078, 0.2667, 0.3176],
         [0.2353, 0.2275, 0.2157,  ..., 0.1882, 0.2431, 0.3137],
         ...,
         [0.8549, 0.8235, 0.7804,  ..., 0.3765, 0.4196, 0.4471],
         [0.7490, 0.7059, 0.6745,  ..., 0.4039, 0.4157, 0.4039],
         [0.7255, 0.6980, 0.6745,  ..., 0.5137, 0.4588, 0.4157]],

        [[0.2235, 0.2118, 0.1961,  ..., 0.2863, 0.3412, 0.3686],
         [0.2196, 0.2118, 0.2000,  ..., 0.2627, 0.3216, 0.3725],
         [0.2196, 0.2118, 0.2000,  ..., 0.2431, 0.2980, 0.3686],
         ...,
         [0.8431, 0.8118, 0.7686,  ..., 0.4627, 0.4980, 0.5255],
         [0.7373, 0.6941, 0.6627,  ..., 0.4863, 0.4980, 0.4863],
         [0.7137, 0.6863, 0.6627,  ..., 0.6039, 0.5373, 0.4941]],

        [[0.2275, 0.2157, 0.2000,  ..., 0.0863, 0.1490, 0.1765],
         [0.2235, 0.2157, 0.2039,  ..., 0.0627, 0.1294, 0.1804],
         [0.2235, 0.2157, 0.2039,  ..., 0.0431, 0.1059, 0

In [17]:
# Save our model for later, so we can train more or make predictions

EPOCH = epochs
# We use the .pt file extension by convention for saving
#    pytorch models
PATH = "model.pt"

# The save function creates a binary storing all our data for us
torch.save({
            'epoch': EPOCH,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            }, PATH)

In [ ]:
# Specify our path
PATH = "model.pt"

# Create a new "blank" model to load our information into
model = FirstNet()

# Recreate our optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Load back all of our data from the file
checkpoint = torch.load(PATH)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
EPOCH = checkpoint['epoch']
